In [ ]:
%pip install langchain langchain-text-splitters PyPDF2

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
from PyPDF2 import PdfFileReader
from io import BytesIO
from snowflake.snowpark import types as T
from snowflake.snowpark.files import SnowflakeFile
from snowflake.snowpark.types import StringType
from snowflake.snowpark.types import StringType, StructField, StructType

from snowflake.snowpark.context import get_active_session
session = get_active_session()

![RAG Architecture](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/RAG_Arch.png)

In [11]:
def readpdf(file_path):
    whole_text = ""
    with SnowflakeFile.open(file_path, 'rb') as file:
        f = BytesIO(file.readall())
        pdf_reader = PdfFileReader(f)
        whole_text = ""
        for page in pdf_reader.pages:
            whole_text += page.extract_text()
    return whole_text

In [12]:
session.sql("USE DATABASE RAG_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("CREATE STAGE IF NOT EXISTS RAG DIRECTORY = (ENABLE = TRUE)").collect()

#Register the UDF. 
session.udf.register(
    func = readpdf
  , return_type = StringType()
  , input_types = [StringType()]
  , is_permanent = True
  , name = 'SNOWPARK_PDF'
  , replace = True
  , packages=['snowflake-snowpark-python','pypdf2','PyCryptodome']
  , stage_location = 'RAG'
)

![Extract Text](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/Extract_Text.png)

In [15]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE RAW_TEXT AS
SELECT
    relative_path
    , file_url
    , snowpark_pdf(build_scoped_file_url(@RAG, relative_path)) as raw_text
from directory(@RAG)

In [16]:
%%sql -r dataframe_2
select * from RAW_TEXT;

In [ ]:
%%sql -r dataframe_3
--Optional : This will fail due to tokens exceeding limit, which means we need to chunk!
SELECT
SNOWFLAKE.CORTEX.COMPLETE('llama2-70b-chat',CONCAT('summarise the following text',raw_text)) 
FROM
RAW_TEXT
LIMIT 1;

A note on chunking
-----
Chunking is the process of splitting a large body of text into smaller 'chunks' whilst attempting to keep as much relevant information as possible. Make the chunks too small and you run the risk of removing key information that the model requires to answer the question. Too large and it may be harder to retreive the correct body of text from the vector search - or spend tokens excessively.

There are many strategies towards chunking. Eg - pass the most relevant, top n relevant chunks, or pass the most relevent chunk + the chunk either side of that one. Play around and see what works for your use case!

![Chunk Text](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/Chunk.text.png)

In [17]:
class text_chunker:

    def process(self,text):        
        text_raw=[]
        text_raw.append(text) 
        
        text_splitter = RecursiveCharacterTextSplitter(
            separators = ["\n"], # Define an appropriate separator. New line is good typically!
            chunk_size = 1000, #Adjust this as you see fit
            chunk_overlap  = 100, #This let's text have some form of overlap. Useful for keeping chunks contextual
            length_function = len,
            add_start_index = True #Optional but useful if you'd like to feed the chunk before/after
        )
    
        chunks = text_splitter.create_documents(text_raw)
        df = pd.DataFrame(chunks, columns=['chunks','meta'])
        
        yield from df.itertuples(index=False, name=None)

In [18]:
#Register the UDTF

schema = StructType([
     StructField("chunk", StringType()),
    StructField("meta", StringType()),
 ])

session.udtf.register( 
    handler = text_chunker,
    output_schema= schema, 
    input_types = [StringType()] , 
    is_permanent = True , 
    name = 'CHUNK_TEXT' , 
    replace = True , 
    packages=['pandas','langchain'], stage_location = 'RAG')

In [20]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE CHUNK_TEXT AS
SELECT
        relative_path,
        func.*
    FROM raw_text AS raw,
         TABLE(chunk_text(raw_text)) as func;

In [ ]:
%%sql -r dataframe_5
select * from chunk_text;

In [ ]:
%%sql -r dataframe_6
ALTER TABLE chunk_text ADD COLUMN vec VECTOR(FLOAT, 768);

![Embed Text](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/Embed_text.png)

![Embedding Functions](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/embedding_functions.png)

In [ ]:
%%sql -r dataframe_7
UPDATE chunk_text SET vec = SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', chunk);

In [ ]:
%%sql -r dataframe_8
select * from chunk_text
limit 5;

In [ ]:
%%sql -r dataframe_9
select snowflake.cortex.complete('mistral-7b','What percent of snowflake customers process unstructured data?') as mixtral_respone;

![RAG Query](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/rag_query.png)

In [ ]:
%%sql -r dataframe_10
select relative_path,chunk, vector_l2_distance(
            snowflake.cortex.embed_text_768('e5-base-v2', 
            'What % of snowflake customers process unstructured data?'
            ), vec
            ) as distance from chunk_text 
            order by distance limit 5;

In [ ]:
%%sql -r dataframe_11
select relative_path,chunk, vector_l2_distance(
            snowflake.cortex.embed_text_768('e5-base-v2', 
            'What data does the carolina panthers collect about me?'
            ), vec
            ) as distance from chunk_text 
            order by distance limit 5;

In [ ]:
%%sql -r dataframe_12
select snowflake.cortex.complete(
    'mistral-7b', 
    concat( 
        'Answer the question based on the context. Be concise.','Context: ',
        (select array_agg(*)::varchar from (
            (select chunk from chunk_text 
            order by vector_l2_distance(
            snowflake.cortex.embed_text_768('e5-base-v2', 
            'What % of snowflake customers process unstructured data?'
            ), vec
            ) limit 5))
            ),
        'Question: ', 
        'What % of snowflake customers process unstructured data?',
        'Answer: '
    )
) as response;

![SiS](https://raw.githubusercontent.com/cromano8/RAG_Snowflake/main/images/sis.png)

In [ ]:
print("Ask Your Data Anything")
print("Built using end-to-end RAG in Snowflake with Cortex functions.\n")

docs = session.table('chunk_text').select('RELATIVE_PATH').distinct().collect()
print("Available documents:", [row[0] for row in docs])

# Configure your query here
doc = docs[0][0] if docs else None
model = 'mistral-7b'
prompt = 'What % of snowflake customers process unstructured data?'

print(f"\nDocument: {doc}")
print(f"Model: {model}")
print(f"Prompt: {prompt}\n")

quest_q = f'''
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    '{model}',
        'In 2 sentences or less, answer the following question: - {prompt}'
) as response
'''

quest_q2 = f'''
select snowflake.cortex.complete(
    '{model}', 
    concat( 
        'Answer the question based on the context. Be concise.','Context: ',
        (select array_agg(*)::varchar from (
            (select chunk from chunk_text 
            where RELATIVE_PATH = '{doc}'
            order by vector_l2_distance(
            snowflake.cortex.embed_text_768('e5-base-v2', 
            '{prompt}'
            ), vec
            ) limit 5))
            ),
        'Question: ', 
        '{prompt}',
        'Answer: '
    )
) as response;
'''

if prompt:
    df_query = session.sql(quest_q).to_pandas()
    print('--- Result without RAG and chunking ---')
    print(df_query['RESPONSE'][0])
    print('\n--- Result with RAG and chunking ---')
    df_query = session.sql(quest_q2).to_pandas()
    print(df_query['RESPONSE'][0])